# Create PWHL Attendance Dataset

[PWHL Data reference](https://github.com/IsabelleLefebvre97/PWHL-Data-Reference)

> Add blockquote



In [44]:
pip install requests

In [45]:
import pandas as pd

In [53]:
import requests
def get_game_summary_df_from_id(game_id):
  game_summary_response = requests.get("https://lscluster.hockeytech.com/feed/index.php?feed=gc&tab=gamesummary&game_id="+str(game_id)+"&key=446521baf8c38984&client_code=pwhl")
  # print(response.status_code)

  game_summary_data = game_summary_response.json()
  game_summary_json = game_summary_data["GC"]["Gamesummary"]["meta"]

  # Load the JSON file into a DataFrame
  game_summary_df = pd.json_normalize(game_summary_json)

  game_summary_df_attendance_df = game_summary_df[["season_id", "home_team", "visiting_team", "game_number", "attendance", "location", "date_played"]]
  return game_summary_df_attendance_df

In [105]:
team_name_map = {
    1: "Boston Fleet",
    2: "Minnesota Frost",
    3: "Montréal Victoire",
    4: "New York Sirens",
    5: "Ottawa Charge",
    6: "Toronto Sceptres",
    8: "Seattle Torrent",
    9: "Vancouver Goldeneyes",
}

primary_location_name_map = {
    29: "Grand Casino Arena",
    30: "Pacific Coliseum",
    1: "TD Place",
    3: "Tsongas Center",
    8: "Place Bell",
    17: "Coca-Cola Coliseum",
    16: "Prudential Center",
    21: "Climate Pledge Arena"
}

all_location_name_map = {
    **primary_location_name_map,
    23: "Ball Arena",
    28: "Agganis Arena",
    # TODO
}


def format_game_summary_df(game_summary_df):
  game_summary_df_formatted = game_summary_df.copy()
  game_summary_df_formatted["home_team"] = game_summary_df_formatted["home_team"].astype(int)
  game_summary_df_formatted["visiting_team"] = game_summary_df_formatted["visiting_team"].astype(int)
  game_summary_df_formatted["season_id"] = game_summary_df_formatted["season_id"].astype(int)
  game_summary_df_formatted["game_number"] = game_summary_df_formatted["game_number"].astype(int)
  game_summary_df_formatted["attendance"] = game_summary_df_formatted["attendance"].astype(int)
  game_summary_df_formatted["location"] = game_summary_df_formatted["location"].astype(int)
  game_summary_df_formatted["home_team_name"] = game_summary_df_formatted["home_team"].map(team_name_map)
  game_summary_df_formatted["visiting_team_name"] = game_summary_df_formatted["visiting_team"].map(team_name_map)
  game_summary_df_formatted["location_name"] = game_summary_df_formatted["location"].map(location_name_map)
  return game_summary_df_formatted


In [103]:
def get_attendance_data_between_ids_inclusive_exclusive(start_game_id, end_game_id):
  primary_venue_dfs = []
  special_venue_dfs = []
  for game_id in range(start_game_id, end_game_id):
    temp_df = get_game_summary_df_from_id(game_id)
    # Only filter for non takeover games
    if len(temp_df) == 1:
      location = temp_df["location"].iloc[0]
      if int(location) in primary_location_name_map:
        primary_venue_dfs.append(temp_df)
      else:
        special_venue_dfs.append(temp_df)
  primary_venue_games = pd.concat(primary_venue_dfs, ignore_index=True)
  special_venue_games = pd.concat(special_venue_dfs, ignore_index=True)
  return format_game_summary_df(primary_venue_games), format_game_summary_df(special_venue_games)

In [95]:
S3_FIRST_GAME_ID = 210
S3_LAST_GAME_ID = 326

In [106]:
primary_venue_attendance_data, special_venue_attendance_data = get_attendance_data_between_ids_inclusive_exclusive(S3_FIRST_GAME_ID, S3_LAST_GAME_ID + 1)

In [107]:
primary_venue_attendance_data

,season_id,home_team,visiting_team,game_number,attendance,location,date_played,home_team_name,visiting_team_name,location_name
0,8,2,6,1,9138,29,2025-11-21,Minnesota Frost,Toronto Sceptres,Grand Casino Arena
1,8,9,8,2,14958,30,2025-11-21,Vancouver Goldeneyes,Seattle Torrent,Pacific Coliseum
2,8,5,4,3,7371,1,2025-11-22,Ottawa Charge,New York Sirens,TD Place
3,8,1,3,4,5166,3,2025-11-23,Boston Fleet,Montréal Victoire,Tsongas Center
4,8,3,4,5,8392,8,2025-11-25,Montréal Victoire,New York Sirens,Place Bell
...,...,...,...,...,...,...,...,...,...,...
87,8,6,4,113,8685,17,2026-04-21,Toronto Sceptres,New York Sirens,Coca-Cola Coliseum
88,8,9,3,114,10946,30,2026-04-21,Vancouver Goldeneyes,Montréal Victoire,Pacific Coliseum
89,8,1,5,115,5215,3,2026-04-22,Boston Fleet,Ottawa Charge,Tsongas Center
90,8,8,2,116,11982,21,2026-04-22,Seattle Torrent,Minnesota Frost,Climate Pledge Arena


In [108]:
special_venue_attendance_data

,season_id,home_team,visiting_team,game_number,attendance,location,date_played,home_team_name,visiting_team_name,location_name
0,8,1,9,11,3516,28,2025-12-03,Boston Fleet,Vancouver Goldeneyes,NaN
1,8,1,2,16,5338,28,2025-12-07,Boston Fleet,Minnesota Frost,NaN
2,8,6,3,18,10438,32,2025-12-17,Toronto Sceptres,Montréal Victoire,NaN
3,8,2,5,24,7238,33,2025-12-21,Minnesota Frost,Ottawa Charge,NaN
4,8,3,6,29,18107,15,2025-12-27,Montréal Victoire,Toronto Sceptres,NaN
5,8,9,2,30,10264,27,2025-12-27,Vancouver Goldeneyes,Minnesota Frost,NaN
6,8,4,8,31,8514,34,2025-12-28,New York Sirens,Seattle Torrent,NaN
7,8,6,8,36,16012,35,2026-01-03,Toronto Sceptres,Seattle Torrent,NaN
8,8,1,9,37,9624,9,2026-01-03,Boston Fleet,Vancouver Goldeneyes,NaN
9,8,1,8,41,6003,28,2026-01-07,Boston Fleet,Seattle Torrent,NaN


In [110]:
average_attendance_by_location = primary_venue_attendance_data.groupby('location_name')['attendance'].mean().reset_index()
print(average_attendance_by_location)

          location_name    attendance
0  Climate Pledge Arena  12875.461538
1    Coca-Cola Coliseum   8379.583333
2    Grand Casino Arena   8142.846154
3      Pacific Coliseum  11228.416667
4            Place Bell   9270.583333
5     Prudential Center   4018.666667
6              TD Place   7298.000000
7        Tsongas Center   4708.857143


In [114]:
all_games = pd.concat([special_venue_attendance_data, primary_venue_attendance_data])

In [115]:
average_attendance_by_team = all_games.groupby('home_team_name')['attendance'].mean().reset_index()
print(average_attendance_by_team)

         home_team_name    attendance
0          Boston Fleet   6560.000000
1       Minnesota Frost   8573.800000
2     Montréal Victoire  10661.066667
3       New York Sirens   6131.466667
4         Ottawa Charge   9197.642857
5       Seattle Torrent  12599.933333
6      Toronto Sceptres   9657.400000
7  Vancouver Goldeneyes  11128.500000


average_attendance_by_team matches the official league attendance report, but my Boston Fleet number is 6560; the official league number is 6,531; I'm not sure where the discrepancy is coming from:

https://lscluster.hockeytech.com/media/pwhl/pwhl/index.php?format=HTML&season_id=8&step=4&sub=15

In [116]:
primary_venue_attendance_data.to_csv("primary_venue_attendance_data.csv")

In [117]:
special_venue_attendance_data.to_csv("special_venue_attendance_data.csv")

In [118]:
all_games.to_csv("pwhl_s3_attendance_data.csv")

In [119]:
average_attendance_by_location.to_csv("primary_venue_average_attendance_by_location.csv")

In [120]:
average_attendance_by_team.to_csv("average_attendance_by_home_team.csv")